# Tutorial 3. Efficient Development

:::{note}
:icon: false

#### Debugging and effective development in VSCode and JupyterLab

When you develop a web app it is important for you to often check that the code you write produces the user interface you expect, the faster you get this feedback, the faster you will develop your app and improve your Panel skills. It's also important for you to be able to use a coding environment you are comfortable with. To this end, Panel offers very good support for Jupyter Notebooks (Classic and JupyterLab) and editors like Visual Studio Code, which are the two environments we will cover in this guide.

## Learning goals

By the end of this tutorial, you should be able to:

- configure Panel correctly for notebook workflows
- use notebook and VS Code feedback loops for faster iteration
- debug served apps with practical VS Code tooling
- catch common Panel bugs before serving, with a static lint and headless smoke tests

:::

## Notebook

### Notebook friendly

Jupyter notebooks have been supported in Panel since day 1 and more generally since the inception of the HoloViz project. Notebooks are environments that allow to iterate quickly, running some code and observing instantaneously its output. Notebooks also happen o be a valid deployment option:

- you can actually develop apps that are meant to live only in a notebook, that you will share as a file with your colleagues and friends. Panel allows you to make your notebooks more user friendly, like IPywidgets does.
- you can serve a notebook with `panel serve file.ipynb`, Panel will parse your notebook to collect all of its code and run it as if it were a Python script.

Jupyter notebooks aren't the only flavor of notebooks available, Panel also supports Visual Studio Code and Google Colab notebooks, and probably works fine on other platforms. For Visual Studio Code notebooks, you will need to install the `jupyter_bokeh` package in your environment.

### `pn.extension`

The first step when working in a notebook environment should **always** be to load the `pn.extension`:

In [ ]:
import panel as pn

pn.extension(throttled=True)

The extension ensures that all required Javascript and CSS resources are added to your notebook environment. If you are going to be using any custom extensions, such as [Vega](https://panel.holoviz.org/reference/panes/Vega.html) or [Tabulator](https://panel.holoviz.org/reference/widgets/Tabulator.html) you must ensure that you initialize these as well. Bokeh is always loaded, so never list `"bokeh"`. Configuration such as `throttled=True`, which makes sliders send their value on release rather than on every drag step, goes in the same call.

In [ ]:
pn.extension('vega', 'tabulator', throttled=True)

### Rich display

One of the major benefits of notebook environments is that they support rich output. This means that if you place an object with rich output at the end of a cell the notebook will figure out how to render the rich representation. Panel uses this mechanism to ensure that all components return a rich representation.

In [ ]:
pane = pn.panel('<marquee>Here is some custom HTML</marquee>')
pane

To instead see a textual representation of the component, you can use the `print` function on any Panel object.

In [ ]:
print(pane)

### Callback logs

When you set up interactivity in your app using one of the various ways Panel offers to register a callback (e.g. `pn.bind`, `@param.depends`, `.param.watch`, `on_<event>` methods), it can be useful for you as a developer to print some debug information. Panel catches all the content sent to *stdout* and *stderr* when callbacks are executed and renders it above the displayed object. The default mode, `'accumulate'`, keeps every message; set `pn.config.console_output` to `'replace'` to only see the last message caught, or to `'disable'` to turn it off. Move the slider in the little app below to see the stream of printed messages being displayed in the notebook.

In [ ]:
pn.config.console_output = 'accumulate'

In [ ]:
def compute(i):
    print(f'compute({i})')
    return '❤️' * i

w = pn.ui.IntSlider(value=1, start=1, end=10, label='Hearts')

pn.Row(w, pn.pane.Markdown(pn.bind(compute, w)))

### *Preview* functionality

With the *Preview* functionality, available in JupyterLab, you can display the app you are building in a JupyterLab tab right next to your code. This is a traditional setup for web developers, who like to quickly visualize the effects of their code changes.

To have your app appear in the *Preview*, you need to mark the objects you want to display with `.servable()`. This is identical to how you would mark objects that you want to serve with `panel serve ...`.

You can enable the *Preview* by clicking on Panel’s logo in the menu bar of your notebook. Once clicked, you should see a new tab being opened next to your notebook tab, and after some moment your app will be rendered in this tab. Just try it out!

In [ ]:
page = pn.ui.Page(title='HM Training!')

def compute(i):
    return '❤️' * i

w_number = pn.ui.IntSlider(value=5, start=1, end=5, label='Hearts')
p_hearts = pn.pane.Markdown(pn.bind(compute, w_number))

page.sidebar.append(w_number)
page.main.append(p_hearts)
page.servable();

:::{tip}

Note the `;` at the end of `page.servable();`. This is a special IPython command that prevents the output of a cell from being displayed. This comes in handy when dealing with a `Page` in a notebook!
:::

This is what you should be able to see after a little while. Now tick `Render on Save` in the *Preview*, change the heart for a ⭐ in the notebook and save it, shortly after the preview will display an updated app.

![alt text](assets/preview.png)

## Visual Studio Code

Visual Studio Code (VS Code) has quickly become one of the most popular Python coding environment. In this section you will see how you can to set yourself a nice developer experience with VS Code when developing a Panel app.

### Setup and the `--dev` server

Save the content of the next cell in a file named `app.py`.

```python
# app.py
import panel as pn

page = pn.ui.Page(title='Loving Panel!')


def compute(i):
    return '❤️' * i


w_number = pn.ui.IntSlider(value=5, start=1, end=5, label='Hearts')
p_hearts = pn.pane.Markdown(pn.bind(compute, w_number))

page.sidebar.append(w_number)
page.main.append(p_hearts)
page.servable()
```

From the VS Code terminal run the following command.

```bash
panel serve app.py --dev --show
```

The `--show` flag will open a browser tab with the live app and the `--dev` flag reloads the app whenever you save a change to the Python source (it replaces the older `--autoreload` flag). Keep this one server running while you work instead of restarting it after each edit: you will see the app being updated live when you save your app file! In the image below the windows have been re-arranged the way web developers like, on one side the code and on the other side a live view of the app, just like the *Preview* functionality in Jupyterlab.

`python app.py` with `.show()` is fine for a one-off preview, but it does not reload, so it is not the loop for iterating.

![alt text](assets/vscode1.png)

### Debugging

While you can debug your app using the often blamed but never outperformed `print`, or by manually adding breakpoints with `breakpoint()`, the VS Code integrated debugger offers a more user friendly experience. 

To configure the integrated debugger for Panel, you will need to add a debugging configuration like the following.

```json
// .vscode/launch.json
{
    "version": "0.2.0",
    "configurations": [
        {
            "name": "panel serve",
            "type": "debugpy",
            "request": "launch",
            "module": "panel",
            "args": [
                "serve",
                "${relativeFile}",
                "--show"
            ],
            "console": "integratedTerminal",
            "justMyCode": true
        }
    ]
}
```

With this configuration in place, launching the debugger will serve the file you are working on with Panel, open a new browser tab pointing at the served app. Interact then with your app to hit one of the breakpoints you set up in VS Code.

![alt text](assets/vscode2.png)

## Catching bugs before you look

Many Panel bugs are *silent*: the app serves, the widgets move, and nothing updates, or it flickers on every change. Clicking around in the browser is the slowest way to find them. A faster loop checks text first and pixels last:

1. **Lint** the source before serving.
2. **Serve** with `--dev` and read the logs after each save.
3. **Smoke test** behavior headlessly by setting widget values in Python.
4. **Look** at the rendered app (layout lint, then a screenshot or your own eyes) once the above are clean.

### Preflight lint

The [holoviz-skills](https://github.com/holoviz-dev/holoviz-skills) package, which teaches AI coding agents these patterns, ships a dependency-free static checker, `preflight.py`, in its Panel skill. It flags the mechanical anti-patterns covered in this training: `@param.depends` methods that return whole layouts (flicker), `.from_param()` widgets created before `super().__init__()`, watchers touching several properties without `pn.io.hold()`, mutated parameter values, radio groups defaulting to `None`, and a missing `throttled=True`.

Run it from wherever the skills are installed, for example `.claude/skills/developing-with-holoviz/skills/panel/scripts/` for Claude Code, or a clone of the repo:

```bash
python <holoviz-skills>/developing-with-holoviz/skills/panel/scripts/preflight.py app.py
```

It exits with `0` and prints `preflight: clean`, or exits with `1` and prints one line per problem:

```
app.py:6: [FROM_PARAM_BEFORE_SUPER] .from_param() widget created before super().__init__() ...
app.py:14: [FLICKER_RECREATED_LAYOUT] `view` is @param.depends (no watch=True) and returns a layout/widget constructor ...
```

### Serving with logs

Capture the server output so you can check for tracebacks after every save:

```bash
panel serve app.py --dev --port 5007 2>&1 | tee panel.log
```

Errors from invalid parameter names or values include the valid options, so read them before guessing.

### Headless smoke tests

Because Panel components are Parameterized objects, you can drive an app from Python without a browser: set a widget's value the way a click would, then assert that the state and the panes updated.

In [ ]:
import param


class HeartCounter(pn.viewable.Viewer):
    hearts = param.Integer(default=3, bounds=(1, 10))

    def __init__(self, **params):
        super().__init__(**params)
        self._slider = pn.ui.IntSlider.from_param(self.param.hearts)
        self._output = pn.pane.Markdown(self._render)
        self._layout = pn.ui.Row(self._slider, self._output)

    @param.depends("hearts")
    def _render(self):
        return '❤️' * self.hearts

    def __panel__(self):
        return self._layout


app = HeartCounter()
app._slider.value = 7
assert app.hearts == 7
assert app._output.object == '❤️' * 7
app

:::{note} Exercise
:class: dropdown

Move the `self._slider = ...` line above `super().__init__(**params)` and rerun the cell. The first assertion still passes, because the widget keeps writing to the parameter, but the second fails: `@param.depends` never fires, so the pane stays stale. This is the classic "widgets move but nothing updates" bug, and `preflight.py` reports it as `FROM_PARAM_BEFORE_SUPER` without running anything.

:::

### Layout lint and screenshots

Once the logs are clean, the same skill folder has `layout_lint.py`, which loads a running app in headless Chromium (it needs `playwright`) at three viewport widths and reports overflow, touch targets under 44px and low-contrast text as text:

```bash
python <holoviz-skills>/developing-with-holoviz/skills/panel/scripts/layout_lint.py http://localhost:5007/app
```

Save screenshots and visual review for what a number can't tell you: hierarchy, spacing and overall feel.

## Recap

You now have a practical development workflow for Panel:

- iterate quickly in notebooks with rich display and Preview
- use VS Code + `panel serve --dev` for app-first iteration
- debug with breakpoints and launch configurations when behavior gets complex
- lint with `preflight.py`, read the server logs, and smoke test headlessly before reaching for the browser

::: {note} References

- [Develop in a notebook](https://panel.holoviz.org/how_to/notebook/index.html)
- [Develop in an editor](https://panel.holoviz.org/how_to/editor/index.html)

:::